# EIA-176 Company Characteristics: Data Exploration

This notebook explores the source data for `core_eia176__yearly_company_characteristics`
(issue #4697), which captures Form EIA-176 Part 3 — Lines A–F — describing company
type, ownership, and operational characteristics.

**Source assets:**
- `_core_eia176__yearly_company_data` — intermediate wide table derived from the NGQS
  custom report (`raw_eia176__numeric_data`). Already contains boolean `is_*` columns
  for operation/ownership type.
- `raw_eia176__operation_types_and_sector_items` — separate extract from the
  `operation_types_and_sector_items` CSV page. May contain overlapping or complementary
  columns for different year ranges.

**Goals:**
1. Identify all Lines A–F columns available in each source.
2. Understand year coverage and null rates per column.
3. Check PK uniqueness `(report_year, operator_id_eia)`.
4. Characterize raw values in boolean/categorical columns (X, 0/1, yes/no, etc.).
5. Verify state normalization path via `core_pudl__codes_subdivisions`.
6. Decide merge strategy and document any quirks for the transform.

In [122]:
import os

assert os.environ.get("DAGSTER_HOME"), (
    "DAGSTER_HOME is not set. Kill the jupyter server, set the env var, and relaunch."
)

In [123]:
import pandas as pd
from dagster import AssetKey

from pudl.etl import defs

## 1. Load source assets

In [124]:
company_data = defs.load_asset_value(AssetKey("_core_eia176__yearly_company_data"))
print(f"_core_eia176__yearly_company_data shape: {company_data.shape}")
print(f"Year range: {company_data['report_year'].min()} – {company_data['report_year'].max()}")
company_data.head(3)

2026-04-20 10:22:09 -0600 - dagster - DEBUG - system - Loading file from: /home/isaac/pudl-work/dagster_home/storage/_core_eia176__yearly_company_data using PickledObjectFilesystemIOManager...


_core_eia176__yearly_company_data shape: (55559, 89)
Year range: 1997 – 2024


,report_year,operating_state,operator_id_eia,operator_name,above_ground_storage_injections_volume,above_ground_storage_withdrawals_volume,alternative_fleet_size,alternative_fuel_fleet_1_yes_0_no,commercial_sales_consumers,commercial_sales_revenue,commercial_sales_volume,commercial_transport_consumers,commercial_transport_revenue,commercial_transport_volume,commercial_volume,customer_choice_residential_eligible,customer_choice_residential_participating,deliveries_out_of_state_volume,disposition_to_distribution_companies_volume,disposition_to_other_pipelines_volume,disposition_to_other_volume,disposition_to_storage_operators_volume,electric_power_sales_consumers,electric_power_sales_revenue,electric_power_sales_volume,electric_power_transport_consumers,electric_power_transport_revenue,electric_power_transport_volume,electric_power_volume,facility_space_heat,heat_content_of_delivered_gas_btu_cf,industrial_sales_consumers,industrial_sales_revenue,industrial_sales_volume,industrial_transport_consumers,industrial_transport_revenue,industrial_transport_volume,industrial_volume,lng_facility_year_end_capacity,lng_facility_year_end_volume,lng_inventory_at_end_of_year_volume,lng_storage_injections_volume,lng_storage_withdrawals_volume,lease_use_volume,losses_from_leaks_volume,marine_terminal_facility_year_end_capacity,marine_terminal_facility_year_end_volume,natural_gas_pump_price,new_pipeline_fill_volume,other,other_receipts_volume,other_sales_consumers,other_sales_revenue,other_sales_volume,other_transport_consumers,other_transport_volume,other_volume,pipeline_dist_storage_compressor_use,production_volume,receipts_at_citygate_delivered_to_sales_customers_cost,receipts_at_citygate_delivered_to_sales_customers_volume,receipts_at_citygate_delivered_to_transportation_customers_volume,receipts_at_citygate_volume,receipts_from_state_or_us_border_volume,residential_sales_consumers,residential_sales_revenue,residential_sales_volume,residential_transport_consumers,residential_transport_revenue,residential_transport_volume,residential_volume,returns_for_repress_reinjection_volume,sales_acquisitions_1_yes_0_no,supplemental_gaseous_fuels_volume,synthetic_production_volume,total_disposition_volume,total_supply_volume,unaccounted_for,underground_storage_injections_volume,underground_storage_withdrawals_volume,vaporization_liquefaction_lng_fuel,vehicle_fuel_sales_consumers,vehicle_fuel_sales_revenue,vehicle_fuel_sales_volume,vehicle_fuel_transport_consumers,vehicle_fuel_transport_revenue,vehicle_fuel_transport_volume,vehicle_fuel_volume,vehicle_fuel_used_in_company_fleet
0,1997,alabama,17600048AL,florala gas department,NaN,NaN,NaN,NaN,50.0,69956.0,8358.0,NaN,NaN,NaN,8358.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28032.0,625.0,251371.0,29469.0,NaN,NaN,NaN,29469.0,NaN,NaN,NaN,NaN,37827.0,28032.0,-9795.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997,alabama,17600049AL,west jefferson gas sys town of,NaN,NaN,NaN,NaN,22.0,27135.0,4606.0,NaN,NaN,NaN,4606.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1018.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34039.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,501.0,221993.0,28016.0,NaN,NaN,NaN,28016.0,NaN,NaN,NaN,NaN,32622.0,34039.0,1417.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997,alabama,17600139AL,spire alabama,NaN,NaN,NaN,NaN,35652.0,72340233.0,9951817.0,87.0,0.0,11033666.0,20985483.0,NaN,NaN,NaN,NaN,NaN,NaN,7800144.0,NaN,NaN,NaN,3.0,0.0,7028200.0,7028200.0,179045.0,1017.0,1498.0,16947052.0,2724147.0,194.0,0.0,48553473.0,51277620.0,NaN,NaN,NaN,783604.0,404450.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,117007347.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,423130.0,246105034.0,29308019.0,NaN,NaN,NaN,29308019.0,NaN,NaN,NaN,NaN,117362115.0,117411797.0,49682.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [125]:
op_types = defs.load_asset_value(AssetKey("raw_eia176__operation_types_and_sector_items"))
print(f"raw_eia176__operation_types_and_sector_items shape: {op_types.shape}")
print(f"Year range: {op_types['report_year'].min()} – {op_types['report_year'].max()}")
op_types.head(3)

2026-04-20 10:22:09 -0600 - dagster - DEBUG - system - Loading file from: /home/isaac/pudl-work/dagster_home/storage/raw_eia176__operation_types_and_sector_items using PickledObjectFilesystemIOManager...


raw_eia176__operation_types_and_sector_items shape: (55589, 72)
Year range: 1997.0 – 2024.0


,commercial_sales_customers,commercial_sales_price_dollars,commercial_sales_revenue_dollars,commercial_sales_volume_mcf,commercial_total_customers,commercial_total_volume_mcf,commercial_transported_customers,commercial_transported_price_dollars,commercial_transported_revenue_dollars,commercial_transported_volume_mcf,electric_sales_customers,electric_sales_price_dollars,electric_sales_revenue_dollars,electric_sales_volume_mcf,electric_total_customers,electric_total_volume_mcf,electric_transported_customers,electric_transported_price_dollars,electric_transported_revenue_dollars,electric_transported_volume_mcf,industrial_sales_customers,industrial_sales_price_dollars,industrial_sales_revenue_dollars,industrial_sales_volume_mcf,industrial_total_customers,industrial_total_volume_mcf,industrial_transported_customers,industrial_transported_price_dollars,industrial_transported_revenue_dollars,industrial_transported_volume_mcf,is_distribution_company_cooperative,is_distribution_company_investor_owned,is_distribution_company_municipally_owned,is_distribution_company_privately_owned,is_gatherer,is_interstate_pipeline,is_intrastate_pipeline,is_liquid_natural_gas_marine_terminal,is_liquid_natural_gas_peak_facility_operator,is_other_ownership,is_other_ownership_2,is_producer,is_public_compressed_natural_gas_fueling_station,is_public_liquid_natural_gas_fueling_station,is_storage_operator,is_synthetic_natural_gas_plant_operator,losses_from_within_report_state_mcf,operating_state,operator_id_eia,operator_name,other_ownership_description,report_year,residential_sales_customers,residential_sales_price_dollars,residential_sales_revenue_dollars,residential_sales_volume_mcf,residential_total_customers,residential_total_volume_mcf,residential_transported_customers,residential_transported_price_dollars,residential_transported_revenue_dollars,residential_transported_volume_mcf,vehicle_fuel_sales_customers,vehicle_fuel_sales_price_dollars,vehicle_fuel_sales_revenue_dollars,vehicle_fuel_sales_volume_mcf,vehicle_fuel_total_customers,vehicle_fuel_total_volume_mcf,vehicle_fuel_transported_customers,vehicle_fuel_transported_price_dollars,vehicle_fuel_transported_revenue_dollars,vehicle_fuel_transported_volume_mcf
0,207.0,1.96,689111.0,352343.0,207.0,352343.0,NaN,0.0,NaN,NaN,1.0,0.32,207398.0,640119.0,1.0,640119.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AK,17602003AK,BARROW UTILITIES ELECTRIC COOP INC,NaN,1997.0,1065.0,2.36,454036.0,192778.0,1065.0,192778.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
1,3.0,0.35,440073.0,1266493.0,3.0,1266493.0,NaN,0.0,NaN,NaN,NaN,0.00,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN,NaN,AK,17602009AK,TIKIGAQ/CONAM,NaN,1997.0,NaN,0.00,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
2,NaN,0.00,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,0.00,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AK,17602016AK,ALASKA PIPELINE COMPANY,NaN,1997.0,NaN,0.00,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN


In [126]:
subdivisions = defs.load_asset_value(AssetKey("core_pudl__codes_subdivisions"))
print(f"core_pudl__codes_subdivisions shape: {subdivisions.shape}")

core_pudl__codes_subdivisions shape: (69, 11)


## 2. Identify Lines A–F columns in each source

EIA-176 Part 3 Lines A–F from the survey form:
- **A** — Type of natural gas (conventional/unconventional) — *requires archiver work, skip for now*
- **B** — Type of business operations (distribution, pipeline, producer, etc.) → `is_*` boolean columns
- **C** — Ownership type (investor-owned, cooperative, municipal, private, other) → more `is_*` columns
- **D** — Alternative fuel vehicle fleet → `alternative_fuel_fleet_*` column
- **E/F** — Investigate availability below

Key boolean/categorical columns expected in `_core_eia176__yearly_company_data`:

In [127]:
# Inspect all columns of _core_eia176__yearly_company_data
print("All columns:")
for col in sorted(company_data.columns):
    print(f"  {col}")

All columns:
  above_ground_storage_injections_volume
  above_ground_storage_withdrawals_volume
  alternative_fleet_size
  alternative_fuel_fleet_1_yes_0_no
  commercial_sales_consumers
  commercial_sales_revenue
  commercial_sales_volume
  commercial_transport_consumers
  commercial_transport_revenue
  commercial_transport_volume
  commercial_volume
  customer_choice_residential_eligible
  customer_choice_residential_participating
  deliveries_out_of_state_volume
  disposition_to_distribution_companies_volume
  disposition_to_other_pipelines_volume
  disposition_to_other_volume
  disposition_to_storage_operators_volume
  electric_power_sales_consumers
  electric_power_sales_revenue
  electric_power_sales_volume
  electric_power_transport_consumers
  electric_power_transport_revenue
  electric_power_transport_volume
  electric_power_volume
  facility_space_heat
  heat_content_of_delivered_gas_btu_cf
  industrial_sales_consumers
  industrial_sales_revenue
  industrial_sales_volume
  ind

In [128]:
# Filter to likely Lines A-F columns (is_* prefix + alternative_fuel + ownership)
lines_af_cols = [c for c in company_data.columns if c.startswith("is_") or "alternative_fuel" in c or "other_ownership" in c]
print(f"Lines A-F candidate columns in _core_eia176__yearly_company_data ({len(lines_af_cols)}):")
for c in lines_af_cols:
    print(f"  {c}")

Lines A-F candidate columns in _core_eia176__yearly_company_data (1):
  alternative_fuel_fleet_1_yes_0_no


In [129]:
# Inspect all columns of raw_eia176__operation_types_and_sector_items
print("raw_eia176__operation_types_and_sector_items columns:")
for col in sorted(op_types.columns):
    print(f"  {col}")

raw_eia176__operation_types_and_sector_items columns:
  commercial_sales_customers
  commercial_sales_price_dollars
  commercial_sales_revenue_dollars
  commercial_sales_volume_mcf
  commercial_total_customers
  commercial_total_volume_mcf
  commercial_transported_customers
  commercial_transported_price_dollars
  commercial_transported_revenue_dollars
  commercial_transported_volume_mcf
  electric_sales_customers
  electric_sales_price_dollars
  electric_sales_revenue_dollars
  electric_sales_volume_mcf
  electric_total_customers
  electric_total_volume_mcf
  electric_transported_customers
  electric_transported_price_dollars
  electric_transported_revenue_dollars
  electric_transported_volume_mcf
  industrial_sales_customers
  industrial_sales_price_dollars
  industrial_sales_revenue_dollars
  industrial_sales_volume_mcf
  industrial_total_customers
  industrial_total_volume_mcf
  industrial_transported_customers
  industrial_transported_price_dollars
  industrial_transported_revenue

In [130]:
# Columns in op_types not in company_data (and vice versa)
cd_cols = set(company_data.columns)
ot_cols = set(op_types.columns)
pk_cols = {"report_year", "operator_id_eia", "operating_state", "operator_name"}

only_in_op_types = ot_cols - cd_cols - pk_cols
only_in_company = cd_cols - ot_cols - pk_cols
shared = (cd_cols & ot_cols) - pk_cols

print(f"Only in op_types ({len(only_in_op_types)}): {sorted(only_in_op_types)}")
print(f"Shared non-PK cols ({len(shared)}): {sorted(shared)}")
# Note: only_in_company is large (all the numeric columns) — not printing

Only in op_types (68): ['commercial_sales_customers', 'commercial_sales_price_dollars', 'commercial_sales_revenue_dollars', 'commercial_sales_volume_mcf', 'commercial_total_customers', 'commercial_total_volume_mcf', 'commercial_transported_customers', 'commercial_transported_price_dollars', 'commercial_transported_revenue_dollars', 'commercial_transported_volume_mcf', 'electric_sales_customers', 'electric_sales_price_dollars', 'electric_sales_revenue_dollars', 'electric_sales_volume_mcf', 'electric_total_customers', 'electric_total_volume_mcf', 'electric_transported_customers', 'electric_transported_price_dollars', 'electric_transported_revenue_dollars', 'electric_transported_volume_mcf', 'industrial_sales_customers', 'industrial_sales_price_dollars', 'industrial_sales_revenue_dollars', 'industrial_sales_volume_mcf', 'industrial_total_customers', 'industrial_total_volume_mcf', 'industrial_transported_customers', 'industrial_transported_price_dollars', 'industrial_transported_revenue_do

## 3. Primary key analysis

In [131]:
pk = ["report_year", "operator_id_eia"]

# PK uniqueness in company_data
dupes_cd = company_data[company_data.duplicated(subset=pk, keep=False)]
print(f"_core_eia176__yearly_company_data duplicate PK rows: {len(dupes_cd)}")
if not dupes_cd.empty:
    print(dupes_cd[pk + lines_af_cols[:5]].head(10))

_core_eia176__yearly_company_data duplicate PK rows: 0


In [132]:
# PK uniqueness in op_types
pk_ot = ["report_year", "operator_id_eia"]
dupes_ot = op_types[op_types.duplicated(subset=pk_ot, keep=False)]
print(f"raw_eia176__operation_types_and_sector_items duplicate PK rows: {len(dupes_ot)}")
if not dupes_ot.empty:
    print(dupes_ot.head(10))

raw_eia176__operation_types_and_sector_items duplicate PK rows: 0


In [133]:
# Year coverage: how many operators per year in each source
cd_yearly = company_data.groupby("report_year").size().rename("company_data")
ot_yearly = op_types.groupby("report_year").size().rename("op_types")
pd.concat([cd_yearly, ot_yearly], axis=1)

,company_data,op_types
report_year,,
1997,1855,1855
1998,1815,1815
1999,1807,1807
2000,1885,1885
2001,1964,1964
2002,1949,1949
2003,1962,1963
2004,1932,1932
2005,1945,1945


## 4. Null rates and value distributions

In [134]:
# Null rates for Lines A-F columns in company_data
null_rates = company_data[lines_af_cols].isna().mean().sort_values(ascending=False)
print("Null rates for Lines A-F columns in _core_eia176__yearly_company_data:")
print(null_rates.to_string())

Null rates for Lines A-F columns in _core_eia176__yearly_company_data:
alternative_fuel_fleet_1_yes_0_no    0.967962


In [135]:
# Raw value distributions for all Lines A-F columns — looking for X, 0/1, yes/no, etc.
for col in lines_af_cols:
    vc = company_data[col].value_counts(dropna=False)
    print(f"\n--- {col} ---")
    print(vc.head(10).to_string())


--- alternative_fuel_fleet_1_yes_0_no ---
alternative_fuel_fleet_1_yes_0_no
NaN    53779
1.0     1780


In [136]:
# Same for op_types columns (excluding shared PK cols)
ot_value_cols = [c for c in op_types.columns if c not in pk_cols]
for col in ot_value_cols:
    vc = op_types[col].value_counts(dropna=False)
    print(f"\n--- {col} (op_types) ---")
    print(vc.head(10).to_string())


--- commercial_sales_customers (op_types) ---
commercial_sales_customers
NaN     20437
1.0       347
20.0      258
21.0      256
3.0       256
4.0       252
17.0      243
18.0      235
25.0      234
19.0      230

--- commercial_sales_price_dollars (op_types) ---
commercial_sales_price_dollars
0.00     20525
8.50        70
9.00        67
8.00        61
10.00       60
7.00        57
8.29        56
8.72        56
8.94        56
8.48        55

--- commercial_sales_revenue_dollars (op_types) ---
commercial_sales_revenue_dollars
NaN         20486
0.0            44
22281.0         5
263495.0        5
42000.0         5
23598.0         4
1000.0          4
30000.0         4
14000.0         4
65120.0         4

--- commercial_sales_volume_mcf (op_types) ---
commercial_sales_volume_mcf
NaN       20475
0.0          49
5000.0       13
6000.0       10
1705.0        8
7000.0        7
7400.0        7
720.0         6
1134.0        6
2000.0        6

--- commercial_total_customers (op_types) ---
comme

## 5. Year-by-year availability of boolean columns

Some boolean columns were added in later years. Check which columns are entirely null in
earlier years.

In [137]:
# For each Lines A-F column, show first and last year with a non-null value
for col in lines_af_cols:
    non_null = company_data.dropna(subset=[col])
    if non_null.empty:
        print(f"{col}: entirely null")
    else:
        print(f"{col}: {int(non_null['report_year'].min())}–{int(non_null['report_year'].max())} ({len(non_null)} rows)")

alternative_fuel_fleet_1_yes_0_no: 2005–2015 (1780 rows)


## 6. State normalization

The `operating_state` column contains full names (e.g., "texas"). We normalize to
two-letter subdivision codes via `core_pudl__codes_subdivisions`, reusing the existing
`_normalize_operating_states` helper.

In [138]:
# What does operating_state look like?
print("Unique operating_state values in company_data (sample):")
print(sorted(company_data["operating_state"].dropna().unique())[:30])

Unique operating_state values in company_data (sample):
['alabama', 'alaska', 'arizona', 'arkansas', 'california', 'colorado', 'connecticut', 'delaware', 'district of columbia', 'fed. gulf of mexico', 'florida', 'georgia', 'hawaii', 'idaho', 'illinois', 'indiana', 'iowa', 'kansas', 'kentucky', 'louisiana', 'maine', 'maryland', 'massachusetts', 'mexico', 'michigan', 'minnesota', 'mississippi', 'missouri', 'montana', 'nebraska']


In [139]:
# Build normalization map and check coverage
from pudl.transform.eia176 import DROP_OPERATING_STATES, _normalize_operating_states

codes = (
    subdivisions.assign(key=lambda d: d["subdivision_name"].str.strip().str.casefold())
    .drop_duplicates("key")
    .set_index("key")["subdivision_code"]
)

norm = company_data["operating_state"].astype(str).str.strip().str.casefold()
not_covered = norm[~norm.isin(codes.index) & ~norm.isin(DROP_OPERATING_STATES)].unique()
print(f"operating_state values not in subdivision codes or DROP list: {not_covered}")

operating_state values not in subdivision codes or DROP list: []


In [140]:
# How many rows would be dropped by null operating_state after normalization?
df_norm = _normalize_operating_states(subdivisions, company_data[["report_year", "operator_id_eia", "operating_state"]])
null_state_rows = df_norm["operating_state"].isna().sum()
total_rows = len(df_norm)
print(f"Rows with null operating_state after normalization: {null_state_rows} / {total_rows} ({null_state_rows/total_rows:.1%})")

Rows with null operating_state after normalization: 27 / 55559 (0.0%)


## 7. Merge strategy: company_data vs. op_types

Decide whether to pull boolean columns from `_core_eia176__yearly_company_data` directly
(simpler, already cleaned) or join with `raw_eia176__operation_types_and_sector_items`
(potentially more complete for certain years or additional columns).

In [141]:
# Check: for shared columns, do company_data and op_types agree?
# Merge on (report_year, operator_id_eia) and compare shared non-PK columns
shared_cols = list(shared)  # defined above

if shared_cols:
    merge_key = ["report_year", "operator_id_eia"]
    merged = company_data[merge_key + shared_cols].merge(
        op_types[merge_key + shared_cols],
        on=merge_key,
        suffixes=("_cd", "_ot"),
        how="inner",
    )
    for col in shared_cols:
        cd_col, ot_col = f"{col}_cd", f"{col}_ot"
        if cd_col in merged.columns:
            mismatches = (merged[cd_col] != merged[ot_col]).sum()
            print(f"{col}: {mismatches} mismatches between sources")
else:
    print("No shared non-PK columns between sources — merge would only add new columns.")

No shared non-PK columns between sources — merge would only add new columns.


In [142]:
# How many op_types rows have no matching row in company_data (and vice versa)?
merge_key = ["report_year", "operator_id_eia"]
cd_keys = company_data[merge_key].drop_duplicates()
ot_keys = op_types[merge_key].drop_duplicates()

only_cd = cd_keys.merge(ot_keys, how="left", indicator=True).query('_merge == "left_only"')
only_ot = ot_keys.merge(cd_keys, how="left", indicator=True).query('_merge == "left_only"')

print(f"Rows in company_data with no op_types match: {len(only_cd)}")
print(f"Rows in op_types with no company_data match: {len(only_ot)}")

if not only_ot.empty:
    print("\nop_types-only rows by year:")
    print(only_ot["report_year"].value_counts().sort_index())

Rows in company_data with no op_types match: 6
Rows in op_types with no company_data match: 36

op_types-only rows by year:
report_year
2003.0    1
2010.0    1
2012.0    1
2013.0    1
2014.0    1
2015.0    1
2016.0    1
2017.0    3
2018.0    2
2019.0    4
2020.0    2
2021.0    3
2022.0    2
2023.0    6
2024.0    7
Name: count, dtype: int64


## 8. Boolean conversion candidates

Identify which columns need `X`/yes-no → `True`/`False` conversion vs. which are already
0/1 floats (from the numeric NGQS custom report).

In [143]:
for col in lines_af_cols:
    unique_vals = set(company_data[col].dropna().astype(str).str.strip().str.lower().unique())
    dtype = company_data[col].dtype
    print(f"{col} [{dtype}]: {unique_vals}")

alternative_fuel_fleet_1_yes_0_no [float64]: {'1.0'}


## 9. Proposed final column set

Select the columns to include in `core_eia176__yearly_company_characteristics`, excluding
Line A (pending archiver work) and any columns found to be entirely null or unavailable.

In [144]:
PRIMARY_KEY = ["report_year", "operator_id_eia"]

# Proposed characteristics columns — adjust after exploring above
CHARS_COLS = [
    # Line B: type of business operations
    "is_distribution_company_cooperative",
    "is_distribution_company_investor_owned",
    "is_distribution_company_municipally_owned",
    "is_distribution_company_privately_owned",
    "is_gatherer",
    "is_interstate_pipeline",
    "is_intrastate_pipeline",
    "is_liquid_natural_gas_marine_terminal",
    "is_liquid_natural_gas_peak_facility_operator",
    "is_producer",
    "is_storage_operator",
    "is_synthetic_natural_gas_plant_operator",
    # Line C: ownership
    "is_other_ownership",
    "other_ownership_description",
    # Line D: alternative fuel fleet (present in some years)
    # (discover exact column name from exploration above)
    # CNG/LNG fueling stations (added ~2016)
    "is_public_compressed_natural_gas_fueling_station",
    "is_public_liquid_natural_gas_fueling_station",
]

# Filter to columns actually present
available_chars = [c for c in CHARS_COLS if c in company_data.columns]
missing_chars = [c for c in CHARS_COLS if c not in company_data.columns]
print(f"Available: {available_chars}")
print(f"Missing (check op_types): {missing_chars}")

Available: []
Missing (check op_types): ['is_distribution_company_cooperative', 'is_distribution_company_investor_owned', 'is_distribution_company_municipally_owned', 'is_distribution_company_privately_owned', 'is_gatherer', 'is_interstate_pipeline', 'is_intrastate_pipeline', 'is_liquid_natural_gas_marine_terminal', 'is_liquid_natural_gas_peak_facility_operator', 'is_producer', 'is_storage_operator', 'is_synthetic_natural_gas_plant_operator', 'is_other_ownership', 'other_ownership_description', 'is_public_compressed_natural_gas_fueling_station', 'is_public_liquid_natural_gas_fueling_station']


In [145]:
# Build a draft characteristics table from company_data alone
df_chars = company_data[PRIMARY_KEY + ["operating_state"] + available_chars].copy()
print(f"Draft shape: {df_chars.shape}")
print(f"Duplicate PK rows: {df_chars.duplicated(subset=PRIMARY_KEY).sum()}")
df_chars.head()

Draft shape: (55559, 3)
Duplicate PK rows: 0


,report_year,operator_id_eia,operating_state
0,1997,17600048AL,alabama
1,1997,17600049AL,alabama
2,1997,17600139AL,alabama
3,1997,17600141AL,alabama
4,1997,17600162AL,alabama


## 10. Smoke test: apply transforms

Once `core_eia176__yearly_company_characteristics` is implemented in
`src/pudl/transform/eia176.py`, load and validate it here.

In [146]:
# Uncomment after the transform asset is implemented:
# result = defs.load_asset_value(AssetKey("core_eia176__yearly_company_characteristics"))
# print(f"Shape: {result.shape}")
# print(result.dtypes)
# print(f"Duplicate PK: {result.duplicated(subset=PRIMARY_KEY).sum()}")
# result.head()

In [147]:
# Row counts per year (for dbt etl_full_row_counts.csv)
# Uncomment after transform is wired:
# result.groupby("report_year").size().rename("n_rows")

## 11. Non-standard null check in op_types boolean columns

Raw CSVs sometimes encode false as `"-"`, `"0"`, `"n"`, empty string, or whitespace.
Check every `is_*` column and `other_ownership_description` for unexpected values.

In [148]:
IS_COLS = [c for c in op_types.columns if c.startswith("is_")]
TEXT_COLS = ["other_ownership_description"]

# For boolean columns: anything that is not "X" or NaN is suspicious
print("=== Unexpected values in is_* columns ===")
for col in IS_COLS:
    unexpected = op_types[col].dropna()
    unexpected = unexpected[unexpected != "X"]
    if not unexpected.empty:
        print(f"{col}: {unexpected.value_counts().to_dict()}")
    else:
        print(f"{col}: OK (only X / NaN)")

print("\n=== other_ownership_description sample values ===")
print(op_types["other_ownership_description"].value_counts(dropna=False).head(20))

=== Unexpected values in is_* columns ===
is_distribution_company_cooperative: OK (only X / NaN)
is_distribution_company_investor_owned: OK (only X / NaN)
is_distribution_company_municipally_owned: OK (only X / NaN)
is_distribution_company_privately_owned: OK (only X / NaN)
is_gatherer: OK (only X / NaN)
is_interstate_pipeline: OK (only X / NaN)
is_intrastate_pipeline: OK (only X / NaN)
is_liquid_natural_gas_marine_terminal: OK (only X / NaN)
is_liquid_natural_gas_peak_facility_operator: OK (only X / NaN)
is_other_ownership: OK (only X / NaN)
is_other_ownership_2: OK (only X / NaN)
is_producer: OK (only X / NaN)
is_public_compressed_natural_gas_fueling_station: OK (only X / NaN)
is_public_liquid_natural_gas_fueling_station: OK (only X / NaN)
is_storage_operator: OK (only X / NaN)
is_synthetic_natural_gas_plant_operator: OK (only X / NaN)

=== other_ownership_description sample values ===
other_ownership_description
NaN                           54821
1.0                              75

## 12. Clarify `is_other_ownership_2`

The form has one 'other' ownership checkbox. Investigate whether `is_other_ownership_2`
is a second checkbox or a data artifact.

In [149]:
# How often are both is_other_ownership and is_other_ownership_2 marked?
both = op_types[(op_types["is_other_ownership"] == "X") & (op_types["is_other_ownership_2"] == "X")]
only_2 = op_types[(op_types["is_other_ownership"].isna()) & (op_types["is_other_ownership_2"] == "X")]
only_1 = op_types[(op_types["is_other_ownership"] == "X") & (op_types["is_other_ownership_2"].isna())]

print(f"Both is_other_ownership AND is_other_ownership_2 marked X: {len(both)}")
print(f"Only is_other_ownership_2 marked X (is_other_ownership NaN): {len(only_2)}")
print(f"Only is_other_ownership marked X (is_other_ownership_2 NaN): {len(only_1)}")

print("\nYear distribution of 'only_2' rows:")
print(only_2["report_year"].value_counts().sort_index())

print("\nSample 'only_2' rows (operator + description):")
print(only_2[["report_year", "operator_id_eia", "operator_name", "is_other_ownership", "is_other_ownership_2", "other_ownership_description"]].head(10).to_string())

Both is_other_ownership AND is_other_ownership_2 marked X: 0
Only is_other_ownership_2 marked X (is_other_ownership NaN): 27
Only is_other_ownership marked X (is_other_ownership_2 NaN): 455

Year distribution of 'only_2' rows:
report_year
2016.0    27
Name: count, dtype: int64

Sample 'only_2' rows (operator + description):
       report_year operator_id_eia operator_name is_other_ownership is_other_ownership_2 other_ownership_description
37611       2016.0      17677205IN  CLEAN ENERGY                NaN                    X                     Cng Lng
37701       2016.0      17677763KS  CLEAN ENERGY                NaN                    X                     Cng Lng
37790       2016.0      17677209KY  CLEAN ENERGY                NaN                    X                     Cng Lng
37947       2016.0      17677212LA  CLEAN ENERGY                NaN                    X                     Cng Lng
37977       2016.0      17674920MA  CLEAN ENERGY                NaN                    X 

In [150]:
# Investigate the 75 rows where other_ownership_description is 1.0 (float)
float_desc = op_types[op_types["other_ownership_description"] == 1.0]
print(f"Rows with other_ownership_description == 1.0: {len(float_desc)}")
print(f"Year range: {float_desc['report_year'].min():.0f}–{float_desc['report_year'].max():.0f}")
print(f"\nYear distribution:")
print(float_desc["report_year"].value_counts().sort_index())
print(f"\nWhich is_* columns are marked for these rows:")
IS_COLS = [c for c in op_types.columns if c.startswith('is_')]
print(float_desc[IS_COLS].apply(lambda c: (c == 'X').sum()).sort_values(ascending=False))
print(f"\nSample rows:")
print(float_desc[["report_year", "operator_id_eia", "operator_name", "operating_state",
                  "is_other_ownership", "is_other_ownership_2", "other_ownership_description"]].head(15).to_string())

Rows with other_ownership_description == 1.0: 75
Year range: 2012–2015

Year distribution:
report_year
2012.0    11
2013.0    12
2014.0     9
2015.0    43
Name: count, dtype: int64

Which is_* columns are marked for these rows:
is_liquid_natural_gas_marine_terminal               32
is_liquid_natural_gas_peak_facility_operator        12
is_interstate_pipeline                               6
is_distribution_company_municipally_owned            5
is_storage_operator                                  3
is_distribution_company_investor_owned               2
is_gatherer                                          1
is_intrastate_pipeline                               1
is_other_ownership                                   1
is_distribution_company_privately_owned              0
is_distribution_company_cooperative                  0
is_other_ownership_2                                 0
is_producer                                          0
is_public_compressed_natural_gas_fueling_station     0
is

## 13. Build merged draft table

Left-join `op_types` (primary source, all `is_*` columns) with the two columns
we need from `_core_eia176__yearly_company_data`: `operating_state` and
`alternative_fuel_fleet_1_yes_0_no`.

In [151]:
PK = ["report_year", "operator_id_eia"]
IS_COLS = [c for c in op_types.columns if c.startswith("is_")]
# operating_state in op_types is already 2-letter postal codes — use it directly
# Only pull alternative_fuel_fleet_1_yes_0_no from company_data
OP_KEEP = PK + ["operator_name", "operating_state"] + IS_COLS + ["other_ownership_description"]
CD_KEEP = PK + ["alternative_fuel_fleet_1_yes_0_no"]

df = op_types[OP_KEEP].merge(
    company_data[CD_KEEP],
    on=PK,
    how="left",
    validate="1:1",
)

print(f"Merged shape: {df.shape}")
print(f"Expected: ({len(op_types)}, {len(OP_KEEP) + len(CD_KEEP) - len(PK)})")
print(f"Duplicate PK rows: {df.duplicated(subset=PK).sum()}")
print(f"Rows where operating_state is null: {df['operating_state'].isna().sum()}")
print(f"\nSample operating_state values (op_types sourced):")
print(df['operating_state'].value_counts().head(10))
df.head(3)

Merged shape: (55589, 22)
Expected: (55589, 22)
Duplicate PK rows: 0
Rows where operating_state is null: 0

Sample operating_state values (op_types sourced):
operating_state
TX    6101
LA    4630
TN    3089
IL    2674
KS    2616
AL    2592
GA    2528
OK    2382
KY    2362
MS    2207
Name: count, dtype: int64


,report_year,operator_id_eia,operator_name,operating_state,is_distribution_company_cooperative,is_distribution_company_investor_owned,is_distribution_company_municipally_owned,is_distribution_company_privately_owned,is_gatherer,is_interstate_pipeline,is_intrastate_pipeline,is_liquid_natural_gas_marine_terminal,is_liquid_natural_gas_peak_facility_operator,is_other_ownership,is_other_ownership_2,is_producer,is_public_compressed_natural_gas_fueling_station,is_public_liquid_natural_gas_fueling_station,is_storage_operator,is_synthetic_natural_gas_plant_operator,other_ownership_description,alternative_fuel_fleet_1_yes_0_no
0,1997.0,17602003AK,BARROW UTILITIES ELECTRIC COOP INC,AK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997.0,17602009AK,TIKIGAQ/CONAM,AK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
2,1997.0,17602016AK,ALASKA PIPELINE COMPANY,AK,NaN,NaN,NaN,NaN,NaN,NaN,X,X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 14. Apply transforms

1. `"X" → True`, `NaN → False` for all `is_*` columns  
2. Rename and convert `alternative_fuel_fleet_1_yes_0_no → has_alternative_fuel_fleet`  
3. Normalize `operating_state` to two-letter codes  
4. Drop rows where `operating_state` is null after normalization

In [152]:
df_t = df.copy()

# 1. Convert X-encoded boolean columns
for col in IS_COLS:
    df_t[col] = df_t[col].eq("X")

# 2. Rename and convert alternative fuel fleet
df_t = df_t.rename(columns={"alternative_fuel_fleet_1_yes_0_no": "has_alternative_fuel_fleet"})
df_t["has_alternative_fuel_fleet"] = df_t["has_alternative_fuel_fleet"].eq(1.0)

# 3. operating_state from op_types is already 2-letter codes — no normalization needed
# Drop rows with null operating_state (operators with no state recorded)
n_before = len(df_t)
df_t = df_t.dropna(subset=["operating_state"])
print(f"Dropped {n_before - len(df_t)} rows with null operating_state")
print(f"Final shape: {df_t.shape}")

# Confirm dtypes
bool_cols = IS_COLS + ["has_alternative_fuel_fleet"]
print("\nBoolean column dtypes:")
print(df_t[bool_cols].dtypes.value_counts())

Dropped 0 rows with null operating_state
Final shape: (55589, 22)

Boolean column dtypes:
bool    17
Name: count, dtype: int64


## 15. Final validation

In [153]:
# PK uniqueness
dupes = df_t.duplicated(subset=PK).sum()
print(f"Duplicate PK rows: {dupes}")
assert dupes == 0, "Unexpected duplicate PKs"

# Boolean columns contain only True/False
for col in bool_cols:
    bad = df_t[col].isna().sum()
    if bad:
        print(f"WARNING: {col} has {bad} NaN values")
assert df_t[bool_cols].isna().sum().sum() == 0, "NaN found in boolean columns"
print("All boolean columns contain only True/False — OK")

# operating_state is all two-letter codes
bad_states = df_t["operating_state"].str.len().ne(2).sum()
print(f"operating_state values with length != 2: {bad_states}")

# True rate per boolean column (to spot columns that are always False)
print("\nTrue rate per boolean column:")
true_rates = df_t[bool_cols].mean().sort_values(ascending=False)
print(true_rates.to_string())

Duplicate PK rows: 0
All boolean columns contain only True/False — OK
operating_state values with length != 2: 0

True rate per boolean column:
is_distribution_company_municipally_owned           0.447948
is_interstate_pipeline                              0.198762
is_distribution_company_investor_owned              0.187501
is_intrastate_pipeline                              0.121805
is_liquid_natural_gas_marine_terminal               0.113817
is_storage_operator                                 0.086924
is_gatherer                                         0.049992
is_distribution_company_privately_owned             0.046160
is_producer                                         0.038551
is_liquid_natural_gas_peak_facility_operator        0.036950
has_alternative_fuel_fleet                          0.032021
is_distribution_company_cooperative                 0.012125
is_other_ownership                                  0.008185
is_public_compressed_natural_gas_fueling_station    0.008095
is

In [154]:
# Row counts per year — will be needed for dbt etl_full_row_counts.csv
print("Row counts per year (for dbt seed):")
print(df_t.groupby("report_year").size().rename("n_rows").to_string())

Row counts per year (for dbt seed):
report_year
1997.0    1855
1998.0    1815
1999.0    1807
2000.0    1885
2001.0    1964
2002.0    1949
2003.0    1963
2004.0    1932
2005.0    1945
2006.0    1941
2007.0    1934
2008.0    1939
2009.0    1998
2010.0    1985
2011.0    2009
2012.0    2016
2013.0    2022
2014.0    2036
2015.0    2043
2016.0    2037
2017.0    2061
2018.0    2066
2019.0    2057
2020.0    2068
2021.0    2033
2022.0    2044
2023.0    2094
2024.0    2091
